# Phase 1: EDA — Bank Account Fraud Dataset (NeurIPS 2022)
**Dataset:** Base variant  
**Source:** Kaggle — sgpjesus/bank-account-fraud-dataset-neurips-2022  
**Goal:** Exploratory Data Analysis before any modeling

In [ ]:
# Install any missing packages (Kaggle already has most of these)
# !pip install pyarrow pandas matplotlib seaborn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

print("Libraries loaded successfully.")

## Step 1: Dataset Description

In [ ]:
# On Kaggle, the dataset is available at this path after you add it
df = pd.read_csv('/kaggle/input/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022/Base.csv')

print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

In [ ]:
# Preview the first few rows
df.head()

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Separate features by type for later use
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove target from feature lists
numerical_cols = [c for c in numerical_cols if c != 'fraud_bool']

print(f"Numerical features ({len(numerical_cols)}): {numerical_cols}")
print(f"\nCategorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"\nTarget: fraud_bool")

## Observations

- The dataset contains **1,000,000 rows and 32 columns**, generated from a
  real-world bank account opening fraud detection system.
- The **target variable** is `fraud_bool` (1 = fraudulent application, 0 = legitimate).
- Features fall into two broad groups: numerical (continuous and count-based)
  and categorical (ordinal and nominal).



## Feature Dictionary

| Feature | Description | Type |
|---|---|---|
| `fraud_bool` | Target variable — 1 if application is fraudulent, 0 if legitimate | Binary |
| `income` | Annual income of the applicant in quantiles. Ranges between [0, 1]. | Numerical |
| `name_email_similarity` | Metric of similarity between email and applicant’s name. Higher values represent higher similarity. Ranges (0-1) | Numerical |
| `prev_address_months_count` | Number of months in previous registered address of the applicant(-1 = not available) | Numerical |
| `current_address_months_count` |Months in currently registered address of the applicant. Ranges between [−1, 406] months (-1 is missing) | Numerical |
| `customer_age` | Applicant’s age in bins per decade (e.g, 20-29 is represented as 20). | Ordinal |
| `days_since_request` | Number of days passed since application was done. Ranges between [0, 78] days. | Numerical |
| `intended_balcon_amount` |Initial transferred amount for application. Ranges between [−1, 108].| Numerical |
| `payment_type` | Credit payment plan type. 5 possible (annonymized) values(AA, AB, AC, AD, AE) | Categorical |
| `zip_count_4w` | Number of applications within same zip code in last 4 weeks. Ranges between [1, 5767]. | Numerical |
| `velocity_6h` | Velocity of total applications made in last 6 hours i.e., average number of applications per hour in the last 6 hrs | Numerical |
| `velocity_24h` | Velocity of total applications made in last 24 hours i.e., average number of applications per hour in the last 24 hrs| Numerical |
| `velocity_4w` | Velocity of total applications made in last 4 weeks, i.e., average number of applications per hour in the last 4 weeks | Numerical |
| `bank_branch_count_8w` | Number of total applications in the selected bank branch in last 8 weeks. Ranges between [0, 2521].| Numerical |
| `date_of_birth_distinct_emails_4w` | Number of emails for applicants with same date of birth in last 4 weeks. Ranges between [0, 42]. | Numerical |
| `employment_status` | Employment status of the applicant. 7 possible (annonymized) values.(CA, CB, CC, CD, CE, CF, CG) | Categorical |
| `credit_risk_score` | Internal score of application risk. Ranges between [−176, 387]. | Numerical |
| `email_is_free` | Domain of application email (either free or paid). | Binary |
| `housing_status` | Current residential status for applicant. 7 possible (annonymized) values(BA, BB, BC, BD, BE, BF, BG) | Categorical |
| `phone_home_valid` |Validity of provided home phone. | Binary |
| `phone_mobile_valid` | Validity of provided mobile phone. | Binary |
| `bank_months_count` | How old is previous account (if held) in months. Ranges between [−1, 31] months (-1 is a missing value). | Numerical |
| `has_other_cards` | If applicant has other cards from the same banking company. | Binary |
| `proposed_credit_limit` |Applicant’s proposed credit limit. Ranges between [200, 2000]. | Numerical |
| `foreign_request` | If origin country of request is different from bank’s country. | Binary |
| `source` | Online source of application. Either browser(INTERNET) or mobile app (APP). | Categorical |
| `session_length_in_minutes` | Length of user session in banking website in minutes. Ranges between [−1, 107] minutes | Numerical |
| `device_os` |Operative system of device that made request. Possible values are: Windows, Macintox, Linux, X11, or other.| Categorical |
| `keep_alive_session` | User option on session logout. | Binary |
| `device_distinct_emails_8w` | Number of distinct emails in banking website from the used device in last 8 weeks. Ranges between [0, 3]. | Numerical |
| `device_fraud_count` | Number of fraud cases previously linked to the same device | Numerical |
| `month` | Month where the application was made. Ranges between [0, 7]. | Ordinal |

**Notes:**
- Features with coded categories (e.g. `employment_status`: CA–CG) are anonymized 
  to protect applicant privacy — the exact meaning of each code is not disclosed.
- `-1` values in `prev_address_months_count` and `bank_months_count` are **sentinel 
  values** representing missing data, not actual measurements.
- `customer_age` and `month` are treated as **ordinal** since they represent ordered 
  bins rather than true continuous values.

## Step 2: Class Distribution


In [ ]:
# Raw counts and percentages
class_counts = df['fraud_bool'].value_counts()
class_pct    = df['fraud_bool'].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    'Count'     : class_counts,
    'Percentage': class_pct.round(2)
})
summary.index = ['Legitimate (0)', 'Fraud (1)']
print(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = ['Legitimate (0)', 'Fraud (1)']
colors = ['#378ADD', '#E24B4A']
counts = [class_counts[0], class_counts[1]]
# --- Bar chart ---
bars = axes[0].bar(labels, counts, color=colors, width=0.5, edgecolor='white')
axes[0].set_title('Class Distribution — Count', fontsize=13, pad=12)
axes[0].set_ylabel('Number of Applications')
axes[0].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{int(x):,}')
)
for bar, count, pct in zip(bars, counts, [class_pct[0], class_pct[1]]):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5000,
        f'{count:,}\n({pct:.2f}%)',
        ha='center', va='bottom', fontsize=11
    )

# --- Pie chart ---
axes[1].pie(
    counts,
    labels=labels,
    colors=colors,
    autopct='%1.2f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops=dict(fontsize=11)
)
axes[1].set_title('Class Distribution — Proportion', fontsize=13, pad=12)

plt.tight_layout()
plt.show()

In [ ]:
# Imbalance ratio: how many legitimate per 1 fraud
ratio = class_counts[0] / class_counts[1]
print(f"Imbalance ratio: {ratio:.1f} legitimate applications for every 1 fraud case")

## Interpretation
The dataset is **heavily imbalanced**. Fraudulent applications account for
roughly **1.1%** of all records, while legitimate ones make up the remaining **~98.9%**.
The imbalance ratio is approximately **89.7:1** (legitimate to fraud).

**Implications for modeling:**
- **Accuracy is misleading**: a model that predicts "legitimate" for every
  application would achieve ~99% accuracy while detecting zero fraud cases.
- **Better metrics to use in later phases**: Precision, Recall, F1-score,
  ROC-AUC, and especially **Average Precision (PR-AUC)**, which is more
  informative under severe class imbalance.
- **Handling strategies** to consider in later phases:
  - Oversampling the minority class (SMOTE)
  - Undersampling the majority class
  - Using `class_weight='balanced'` in sklearn models
  - Threshold tuning on predicted probabilities

## Step 3: Statistical Summary Report


In [ ]:
# Descriptive statistics for all numerical features, transposed for readability
desc = df[numerical_cols].describe().T.round(2)

# Add median as an extra column (describe() calls it '50%')
desc.rename(columns={
    '25%': 'Q1',
    '50%': 'median',
    '75%': 'Q3'
}, inplace=True)

desc

In [ ]:
# Detect sentinel -1 values from the data itself
print("Features containing -1 values:\n")
for col in numerical_cols:
    count = (df[col] == -1).sum()
    if count > 0:
        pct = count / len(df) * 100
        print(f"  {col:<40} → {count:>7,} rows ({pct:.2f}%)")

In [ ]:
# Skewness tells us how asymmetric each distribution is
# |skew| > 1 = highly skewed, worth noting
skewness = df[numerical_cols].skew().sort_values(ascending=False).round(2)

print("Skewness per feature (sorted):\n")
for col, skew in skewness.items():
    flag = " ← highly skewed" if abs(skew) > 1 else ""
    print(f"  {col:40} {skew:>7.2f}{flag}")

In [ ]:
# A large gap between mean and median signals skew or outlier influence
mean_median_gap = (
    (desc['mean'] - desc['median'])
    .abs()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(12, 5))
mean_median_gap.plot(kind='barh', ax=ax, color='#378ADD', edgecolor='white')
ax.set_title('Top 10 features — absolute gap between mean and median', fontsize=13, pad=12)
ax.set_xlabel('|Mean − Median|')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Interpretation

The descriptive statistics reveal several important characteristics:

**Sentinel values (-1):**
Five features clearly use -1 as a sentinel for missing data based on their documented ranges:

| Feature | Missing rows | % of dataset |
|---|---|---|
| `prev_address_months_count` | 712,920 | 71.29% |
| `bank_months_count` | 253,635 | 25.36% |
| `current_address_months_count` | 4,254 | 0.43% |
| `session_length_in_minutes` | 2,015 | 0.20% |
| `device_distinct_emails_8w` | 359 | 0.04% |

`prev_address_months_count` is the most concerning — **71.29% of values are missing**, meaning this feature may have limited usefulness for modeling without careful imputation.

Note: `credit_risk_score` has 488 rows with a value of -1, but since its documented range is [-176, 387], -1 falls within valid score territory and likely represents a legitimate risk score rather than a missing value.

**Suspicious ranges:**
- `velocity_6h` and `velocity_24h` have very high max values relative to their means, indicating extreme outliers in application velocity — likely fraud-related spikes.
- `intended_balcon_amount` minimum of -1 (sentinel) with a max of ~108 suggests most transfer amounts are small or absent.

**Skewness:**
Following the commonly used rule of thumb, features with |skewness| > 1 are considered highly skewed. Several velocity and count features fall into this category, meaning most applicants have low values but a few have extremely high ones. These spikes are likely fraud signals worth investigating in the outlier detection step.

**Mean vs Median (chart above):**

`velocity_6h` has the largest gap (340), followed by `proposed_credit_limit` (315) and `zip_count_4w` (310). These three features are most heavily influenced by outliers. `bank_branch_count_8w` and `velocity_4w` also show notable gaps. Features at the bottom of the chart such as `credit_risk_score` and `intended_balcon_amount` have relatively stable distributions with means close to their medians.

## Step 4: Data Quality Assessment


In [ ]:
# Check for actual NaN missing values (not sentinel -1s)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

missing_cols = missing_df[missing_df['Missing Count'] > 0]

if len(missing_cols) == 0:
    print("No NaN missing values found in the dataset.")
else:
    print(missing_cols)

In [ ]:
# Check for exact duplicate rows
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows: {n_duplicates:,}")
print(f"Percentage of dataset: {n_duplicates / len(df) * 100:.4f}%")

In [ ]:
# Check unique values in categorical columns for unexpected entries
print("Unique values per categorical feature:\n")
for col in categorical_cols:
    vals = df[col].unique()
    print(f"  {col:30} → {sorted(vals)}")

In [ ]:
# Visualize the scale of sentinel -1 missingness across affected features
sentinel_cols = [
    'prev_address_months_count',
    'bank_months_count',
    'current_address_months_count',
    'session_length_in_minutes',
    'device_distinct_emails_8w'
]

sentinel_pct = pd.Series({
    col: (df[col] == -1).sum() / len(df) * 100
    for col in sentinel_cols
}).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(sentinel_pct.index, sentinel_pct.values,
               color='#E24B4A', edgecolor='white')

for bar, val in zip(bars, sentinel_pct.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}%', va='center', fontsize=11)
ax.set_title('Sentinel -1 missingness by feature (% of dataset)', fontsize=13, pad=12)
ax.set_xlabel('% rows with -1')
ax.set_xlim(0, 85)
plt.tight_layout()
plt.show()

## Interpretation

**True NaN missing values:**
The dataset contains no NaN missing values — all fields are filled. However,
this does not mean the data is complete, as missing information is encoded
as sentinel -1 values rather than explicit nulls.

**Duplicate rows:**
No duplicate rows were found

**Categorical consistency:**
All categorical features contain only their expected coded values with no
unexpected entries, typos, or inconsistencies detected.

**Sentinel -1 missingness:**
Five features use -1 to encode missing data. `prev_address_months_count`
is the most severely affected at 71.29%, followed by `bank_months_count`
at 25.36%. The remaining three features have negligible missingness (< 0.5%).

**Plan for later phases:**
- Replace -1 sentinel values with NaN to allow standard imputation techniques.
- For `prev_address_months_count` (71.29% missing): given the very high
  missingness rate, consider dropping this feature instead
  of imputing, as imputing 71% of values introduces significant noise.
- For `bank_months_count` (25.36% missing): median imputation or
  model-based imputation (e.g. KNN) are reasonable strategies.
- For the remaining features with < 0.5% missing: median imputation is sufficient.

## Step 5: Outlier Detection


In [ ]:
# Box plots for all numerical features
# Use a sample for speed
sample = df.sample(10_000, random_state=42)

binary_cols = [
    'email_is_free', 'phone_home_valid', 'phone_mobile_valid',
    'has_other_cards', 'foreign_request', 'keep_alive_session'
]

continuous_cols = [c for c in numerical_cols if c not in binary_cols]

n_cols = 4
n_rows = -(-len(continuous_cols) // n_cols)  # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    sns.boxplot(
        data=sample, y=col, x='fraud_bool',
        hue='fraud_bool',
        palette={0: '#378ADD', 1: '#E24B4A'},
        ax=axes[i], width=0.5,
        legend=False
    )
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('fraud_bool (0=legit, 1=fraud)', fontsize=8)
    axes[i].set_ylabel('')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Box Plots of Numerical Features by Fraud Label',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Interpretation

**Box plots:**
The box plots grouped by `fraud_bool` reveal clear distributional differences
between legitimate (blue) and fraudulent (red) applications in several features.
Binary features were excluded from this plot as box plots are not meaningful
for 0/1 values.

**Features with notable differences:**

- `income`: Fraudulent applicants tend to claim significantly higher income
  (median ~0.75) compared to legitimate ones (median ~0.60). This is likely
  a deliberate strategy to qualify for higher credit limits.

- `name_email_similarity`: Fraudulent applicants have a clearly lower median
  (~0.25) vs legitimate(~0.50), suggesting fraudsters use emails that do not
  match their stated name — a strong and intuitive fraud signal.

- `customer_age`: Fraudulent applicants tend to claim older age (median ~40)
  compared to legitimate applicants (median ~30), possibly to appear more
  financially established and credible.

- `intended_balcon_amount`: Legitimate applicants show a wider spread with
  higher transfer amounts, while fraudulent ones are compressed near 0 —
  fraudsters tend to request little to no initial balance transfer.

- `bank_branch_count_8w`: Legitimate applications come from branches with
  wide activity ranges (up to 2500), while fraudulent ones cluster near 0 —
  suggesting fraudsters deliberately target low-activity branches, possibly
  to avoid detection.

- `credit_risk_score`: One of the clearest visual separations in the dataset.
  Fraudulent applicants have a much higher median score (~200) vs legitimate
  (~110), indicating the bank's internal scoring system is effectively
  capturing fraud risk.

- `bank_months_count`: Fraudulent applicants have very short or no prior
  banking history (median ~0), while legitimate applicants have a median of
  ~5 months. This suggests fraudsters are opening new accounts with no
  established banking relationship.

- `proposed_credit_limit`: Fraudulent applicants request dramatically higher
  credit limits (median ~1500) vs legitimate applicants (median ~450),
  consistent with the income inflation finding — fraudsters overstate income
  to justify requesting higher limits.

- `month`: Fraud applications are slightly more concentrated in later months
  (median ~4) compared to legitimate ones (median ~3), potentially indicating
  a growing fraud trend over the 8-month observation window.

In [ ]:
# Z-score analysis on the full dataset: z = (value - mean) / standard_deviation
# |z| > 3 means the value is more than 3 standard deviations from the mean
from scipy import stats

binary_cols = [
    'email_is_free', 'phone_home_valid', 'phone_mobile_valid',
    'has_other_cards', 'foreign_request', 'keep_alive_session'
]

continuous_cols = [c for c in numerical_cols if c not in binary_cols]

z_scores = df[continuous_cols].apply(
    lambda col: np.abs(stats.zscore(col, nan_policy='omit'))
)

outlier_counts = (z_scores > 3).sum().sort_values(ascending=False)
outlier_pct = (outlier_counts / len(df) * 100).round(2)

outlier_df = pd.DataFrame({
    'Outlier Count': outlier_counts,
    'Outlier %': outlier_pct
})

print("Outliers per feature (|z-score| > 3):\n")
print(outlier_df[outlier_df['Outlier Count'] > 0])

In [ ]:
# For the top 5 most outlier-heavy features, check what fraction
# of their outliers are fraud vs legitimate
top5 = outlier_counts.head(5).index.tolist()

print("Fraud rate among outliers vs non-outliers (z-score):\n")
for col in top5:
    is_outlier = z_scores[col] > 3
    fraud_in_outliers     = df.loc[is_outlier, 'fraud_bool'].mean() * 100
    fraud_in_non_outliers = df.loc[~is_outlier, 'fraud_bool'].mean() * 100
    print(f"  {col}")
    print(f"    Fraud rate in outliers:     {fraud_in_outliers:.2f}%")
    print(f"    Fraud rate in non-outliers: {fraud_in_non_outliers:.2f}%\n")

In [ ]:
# ── IQR Method ──────────────────────────────────────────────────────────────
# More robust for skewed distributions — uses quartiles instead of mean/std
# This is the same logic used by the box plot whiskers above

iqr_outlier_counts = {}

binary_cols = [
    'email_is_free', 'phone_home_valid', 'phone_mobile_valid',
    'has_other_cards', 'foreign_request', 'keep_alive_session'
]

continuous_cols = [c for c in numerical_cols if c not in binary_cols]

for col in continuous_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    is_outlier = (df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)
    iqr_outlier_counts[col] = is_outlier.sum()

iqr_df = pd.DataFrame({
    'IQR Outlier Count': iqr_outlier_counts,
    'IQR Outlier %': {col: round(count / len(df) * 100, 2)
                      for col, count in iqr_outlier_counts.items()}
}).sort_values('IQR Outlier Count', ascending=False)

print("Outliers per feature (IQR method):\n")
print(iqr_df[iqr_df['IQR Outlier Count'] > 0])

In [ ]:
# Fraud rate among outliers vs non-outliers — IQR method
# Using the same top 5 features by IQR outlier count for fair comparison

top5_iqr = iqr_df.head(5).index.tolist()

print("Fraud rate among outliers vs non-outliers (IQR method):\n")
for col in top5_iqr:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    is_outlier = (df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)
    
    fraud_in_outliers     = df.loc[is_outlier, 'fraud_bool'].mean() * 100
    fraud_in_non_outliers = df.loc[~is_outlier, 'fraud_bool'].mean() * 100
    print(f"  {col}")
    print(f"    Outlier count:              {is_outlier.sum():,}")
    print(f"    Fraud rate in outliers:     {fraud_in_outliers:.2f}%")
    print(f"    Fraud rate in non-outliers: {fraud_in_non_outliers:.2f}%\n")

In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────────
# Compare z-score vs IQR outlier counts for features detected by either method

comparison = pd.DataFrame({
    'Z-score %': outlier_pct,
    'IQR %'    : iqr_df['IQR Outlier %']
}).fillna(0).sort_values('IQR %', ascending=False)

comparison = comparison[comparison[['Z-score %', 'IQR %']].sum(axis=1) > 0]

fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(comparison))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], comparison['Z-score %'],
               width, label='Z-score (|z|>3)', color='#378ADD', edgecolor='white')
bars2 = ax.bar([i + width/2 for i in x], comparison['IQR %'],
               width, label='IQR (1.5×IQR)',   color='#E24B4A', edgecolor='white')

ax.set_xticks(list(x))
ax.set_xticklabels(comparison.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Outlier %')
ax.set_title('Outlier Detection: Z-score vs IQR method', fontsize=13, pad=12)
ax.legend()
plt.tight_layout()
plt.show()

## Interpretation

**Why two methods?**
Two outlier detection methods are applied here because they make different
assumptions and suit different data characteristics:

- **Z-score (|z| > 3):** Measures how many standard deviations a value is
  from the mean. Assumes an approximately normal distribution. Sensitive to
  skewness because extreme values inflate the mean and standard deviation,
  which can cause the method to both miss real outliers and flag false ones.

- **IQR (1.5 × IQR):** Uses Q1, Q3, and the interquartile range — all anchored
  to the middle 50% of data, making them unaffected by extreme values. More
  appropriate for skewed features. This is the same logic used by the whiskers
  in the box plots above, making both visually and analytically consistent.

> The threshold |z| > 3 follows the empirical rule of the normal distribution,
> under which only ~0.3% of data falls beyond 3 standard deviations. While our
> features are not perfectly normal, this threshold serves as a practical and
> widely accepted guideline. IQR uses Tukey's rule (1.5 × IQR), which is
> distribution-free and more appropriate for skewed data.

---

**Comparison chart observations:**
The chart confirms the expected pattern: **IQR flags significantly more outliers
than z-score across almost every feature.** This is because most features in
this dataset are right-skewed — skewness inflates the standard deviation used
by z-score, shrinking z-scores for extreme values and causing the method to
undercount outliers. IQR, being robust to skew, captures them more accurately.

The gap is most dramatic in:
- `proposed_credit_limit`: IQR flags 24.17% vs z-score's 0.87%
- `intended_balcon_amount`: IQR flags 22.27% vs z-score's 1.90%
- `prev_address_months_count`: IQR flags 15.73% vs z-score's 2.53%

Features where both methods agree closely (`device_distinct_emails_8w`,
`date_of_birth_distinct_emails_4w`) tend to be the least skewed ones,
confirming that skewness is the primary driver of disagreement between methods.

---

**Z-score fraud rate analysis (top 5 by z-score outlier count):**

| Feature | Fraud in outliers | Fraud in non-outliers | Signal? |
|---|---|---|---|
| `bank_branch_count_8w` | 1.03% | 1.11% |  Noise |
| `device_distinct_emails_8w` | 3.73% | 1.02% | **Strong signal (3.7×)** |
| `prev_address_months_count` | 0.62% | 1.12% | Noise — sentinel distortion |
| `foreign_request` | 2.20% | 1.07% | **Signal (2×)** |
| `session_length_in_minutes` | 2.00% | 1.08% | **Signal (~2×)** |

---

**IQR fraud rate analysis (top 5 by IQR outlier count, binary features excluded):**

| Feature | Outlier count | Fraud in outliers | Fraud in non-outliers | Signal? |
|---|---|---|---|---|
| `proposed_credit_limit` | 241,742 | 2.11% | 0.78% | **Signal (2.7×)** |
| `intended_balcon_amount` | 222,702 | 0.51% | 1.27% | Inverse — outliers less likely fraud |
| `bank_branch_count_8w` | 175,243 | 0.74% | 1.18% | Noise |
| `prev_address_months_count` | 157,320 | 0.37% | 1.24% | Noise — sentinel distortion |
| `days_since_request` | 94,834 | 1.30% | 1.08% | **Marginal — slight enrichment** |

The IQR analysis reveals an important finding that z-score largely missed:
**`proposed_credit_limit`** is a meaningful fraud signal. With 24.17% of its
values flagged as outliers and a fraud rate of 2.11% among them (vs 0.78%
in non-outliers), applicants requesting unusually high credit limits are 2.7×
more likely to be fraudulent — consistent with the box plot observation that
fraudsters request dramatically higher limits. Z-score missed this because
the right-skewed distribution inflated its standard deviation.

`intended_balcon_amount` shows an **inverse relationship** — its outliers are
actually less likely to be fraud (0.51% vs 1.27%). High initial transfer amounts
are associated with legitimate applications, while fraudsters tend to request
little to no initial balance transfer, as confirmed by the box plots.

`days_since_request` shows only marginal fraud enrichment (1.30% vs 1.08%),
not strong enough to be considered a meaningful signal.

---

**Conclusion:**
The two methods together paint a clearer picture than either alone:

- **Meaningful fraud signals** (outliers = higher fraud rate):
  `device_distinct_emails_8w` (3.7×), `proposed_credit_limit` (2.7×),
  `foreign_request` (2×), `session_length_in_minutes` (2×)

- **Noise or distorted** (outliers = same or lower fraud rate):
  `bank_branch_count_8w`, `prev_address_months_count` (sentinel distortion),
  `intended_balcon_amount` (inverse relationship)

For later phases, **IQR is the preferred outlier detection method** for this
dataset given its heavy skewness. Tree-based models such as Random Forest
or XGBoost are also recommended as they are naturally robust to outliers
and do not require removal or capping of extreme values.

## Step 6: Feature Distributions

In [ ]:
# Separate features into meaningful groups for plotting
binary_cols = [
    'email_is_free', 'phone_home_valid', 'phone_mobile_valid',
    'has_other_cards', 'foreign_request', 'keep_alive_session'
]

categorical_cols = [
    'payment_type', 'employment_status',
    'housing_status', 'source', 'device_os'
]

# Continuous: all numerical except binary and ordinal
ordinal_cols = ['customer_age', 'month']

continuous_cols = [
    c for c in numerical_cols
    if c not in binary_cols 
    and c not in ordinal_cols
    and c != 'device_fraud_count'  # near-constant, KDE not meaningful
]

print(f"Continuous : {len(continuous_cols)} features")
print(f"Ordinal    : {ordinal_cols}")
print(f"Binary     : {len(binary_cols)} features")
print(f"Categorical: {categorical_cols}")

In [ ]:
# Replace sentinel -1 values with NaN for visualization only
# Original df is NOT modified — this is purely for cleaner plots
sentinel_cols = [
    'prev_address_months_count',
    'current_address_months_count',
    'bank_months_count',
    'session_length_in_minutes',
    'device_distinct_emails_8w',
    'intended_balcon_amount'
]

sample_clean = sample.copy()
for col in sentinel_cols:
    sample_clean[col] = sample_clean[col].replace(-1, np.nan)

print("Sentinel -1 values replaced with NaN in sample_clean.")
print("Original df is unchanged:", (df[sentinel_cols] == -1).sum().sum(), "sentinel values still in df.")

In [ ]:
# KDE (Kernel Density Estimation) plots split by fraud_bool to show distribution per class
# Sample for performance — KDE is slow on 1M rows
sample = df.sample(20_000, random_state=42)

n_cols = 4
n_rows = -(-len(continuous_cols) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    for label, color in [(0, '#378ADD'), (1, '#E24B4A')]:
        subset = sample_clean[sample_clean['fraud_bool'] == label][col].dropna()
        sns.kdeplot(subset, ax=axes[i], color=color,
                fill=True, alpha=0.3, linewidth=1.5,
                label=f'{"Fraud" if label else "Legit"}')
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('KDE Plots of Continuous Features by Fraud Label',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
#Ordinal features: customer_age and month
# Show count per value split by fraud label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(ordinal_cols):
    fraud_rate = df.groupby(col)['fraud_bool'].mean() * 100
    fraud_rate.plot(kind='bar', ax=axes[i],
                    color='#E24B4A', edgecolor='white', width=0.6)
    axes[i].set_title(f'Fraud rate by {col}', fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Fraud rate (%)')
    axes[i].tick_params(axis='x', rotation=0)
    for bar in axes[i].patches:
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f'{bar.get_height():.2f}%',
            ha='center', va='bottom', fontsize=8
        )

plt.suptitle('Fraud Rate by Ordinal Feature', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features: show fraud rate per category
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    fraud_rate = df.groupby(col)['fraud_bool'].mean().sort_values(ascending=False) * 100
    fraud_rate.plot(kind='bar', ax=axes[i],
                    color='#E24B4A', edgecolor='white', width=0.6)
    axes[i].axhline(y=df['fraud_bool'].mean() * 100,
                    color='black', linestyle='--', linewidth=1,
                    label='Overall fraud rate')
    axes[i].set_title(f'Fraud rate by {col}', fontsize=11)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Fraud rate (%)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(fontsize=8)
    for bar in axes[i].patches:
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f'{bar.get_height():.2f}%',
            ha='center', va='bottom', fontsize=8
        )

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Fraud Rate by Categorical Feature', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Binary features: show fraud rate for value 0 vs 1
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(binary_cols):
    fraud_rate = df.groupby(col)['fraud_bool'].mean() * 100
    fraud_rate.plot(kind='bar', ax=axes[i],
                    color=['#378ADD', '#E24B4A'], edgecolor='white', width=0.5)
    axes[i].axhline(y=df['fraud_bool'].mean() * 100,
                    color='black', linestyle='--', linewidth=1,
                    label='Overall fraud rate')
    axes[i].set_title(f'Fraud rate by {col}', fontsize=11)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Fraud rate (%)')
    axes[i].set_xticklabels(['No (0)', 'Yes (1)'], rotation=0)
    axes[i].legend(fontsize=8)
    for bar in axes[i].patches:
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f'{bar.get_height():.2f}%',
            ha='center', va='bottom', fontsize=8
        )

plt.suptitle('Fraud Rate by Binary Feature', fontsize=14)
plt.tight_layout()
plt.show()

## Interpretation

**Note on `device_fraud_count`:** This feature was excluded from the KDE plots
as it is near-constant — virtually all applicants have a value of 0, making
a KDE curve uninformative. This feature carries almost no variance and may
be a candidate for removal in later phases.

**Note on sentinel values:** Before plotting, sentinel -1 values in
`prev_address_months_count`, `current_address_months_count`,
`bank_months_count`, `session_length_in_minutes`, `device_distinct_emails_8w`,
and `intended_balcon_amount` were replaced with NaN for visualization only.
The original dataset was not modified.

---

### Continuous Features (KDE Plots)

**`income` — Bimodal, fraud skewed high:**
Both classes show a bimodal distribution, but fraudulent applicants have a
sharp dominant peak near 1.0 (maximum income), while legitimate applicants
are more evenly spread across the range. This confirms that fraudsters
systematically overstate income, likely to qualify for higher credit limits.

**`name_email_similarity` — Clear separation, strong signal:**
Legitimate applicants show a roughly uniform distribution across all similarity
values, while fraudulent applicants are strongly concentrated near 0. This is
one of the clearest distributional separations in the dataset — fraudsters
consistently use emails that bear no resemblance to their stated name,
consistent with the use of fake or randomly generated email addresses.

**`days_since_request` — Extremely right-skewed (skewness = 9.28):**
Both classes are heavily concentrated near 0 with a long right tail, confirming
the highest skewness in the dataset. Most applications — fraudulent and
legitimate — are submitted almost immediately. Fraudulent applications are
slightly more concentrated at 0, suggesting fraudsters act faster after
initiating an application.

**`intended_balcon_amount` — Fraud concentrated at zero:**
Legitimate applicants show a secondary bump around 40–60, indicating some
transfer moderate initial amounts. Fraudulent applicants are almost entirely
concentrated at the first peak near 0 — fraudsters rarely initiate any
balance transfer, consistent with the box plot finding.

**`velocity_4w` — Multimodal distribution:**
Both classes exhibit a multimodal distribution with multiple distinct peaks,
suggesting different subgroups within each class. Legitimate applicants show
more pronounced peaks at higher velocity values, while fraud is more
concentrated at lower values — counterintuitive but consistent with the
box plot and outlier analysis.

**`bank_branch_count_8w` — Fraud concentrated near zero:**
Legitimate applications come from branches with a wide range of activity,
while fraudulent applications are sharply concentrated near 0. Fraudsters
overwhelmingly target low-activity branches, likely to reduce the chance
of detection by branch staff familiar with unusual patterns.

**`credit_risk_score` — Strong signal, clear shift:**
One of the strongest KDE separations in the dataset. Legitimate applicants
peak around 100–150, while fraudulent applicants peak significantly higher
at 200–250 with a wider spread. The bank's internal scoring system is
effectively assigning higher risk scores to fraudulent applications.

**`bank_months_count` — Bimodal, strong separation:**
Legitimate applicants show a bimodal distribution with peaks near 0 and ~28,
reflecting a mix of new and long-standing customers. Fraudulent applicants
are almost entirely concentrated at 0 — virtually no prior banking history.
This is one of the most visually striking separations in the entire dataset.

**`proposed_credit_limit` — Multimodal, very strong signal:**
Both distributions are multimodal with peaks at standard credit limit tiers
(~200, ~500, ~1500, ~2000). Legitimate applicants are dominated by the lowest
tier (~200), while fraudulent applicants heavily favor the highest tiers
(~1500–2000). This is the clearest multimodal separation in the dataset and
directly ties to the income inflation pattern — fraudsters overstate income
to justify requesting maximum credit limits.

**Remaining features** (`zip_count_4w`, `velocity_6h`, `velocity_24h`,
`current_address_months_count`, `prev_address_months_count`,
`date_of_birth_distinct_emails_4w`, `session_length_in_minutes`):
These features show largely overlapping distributions between fraud and
legitimate classes with no strong visual separation, suggesting limited
standalone discriminative power.

---

### Ordinal Features

**`customer_age` — Monotonic increase, strong signal:**
Fraud rate increases steadily and consistently with age bin:
10→0.35%, 20→0.49%, 30→0.83%, 40→1.20%, 50→2.00%, 60→3.30%,
70→4.04%, 80→4.93%, 90→5.26%.
Older age bins are dramatically more associated with fraud — fraudsters
tend to claim older ages, possibly to appear more financially established.
The monotonic nature of this trend makes `customer_age` a strong ordinal
predictor for modeling.

**`month` — Growing fraud trend over time:**
Fraud rate starts at 1.13% in month 0, dips slightly through months 1–3
(0.94%–0.92%), then rises steadily to 1.47% by month 7. This upward trend
suggests fraud activity is increasing over the 8-month observation window,
which has important implications — models trained on early months may
underperform on later ones, motivating time-based train/test splits in
later phases.

---

### Categorical Features

**`payment_type` — Notable variation:**
AC has the highest fraud rate at 1.67% (above the 1.11% baseline), while
AE is the lowest at 0.35%. The anonymized categories show meaningful
variation, suggesting payment plan type is associated with fraud risk.

**`employment_status` — Strong signal, wide range:**
CC reaches 2.47% fraud rate (more than double the baseline), while CF and
CE are at 0.19% and 0.23% respectively. The wide range across employment
categories (0.19%–2.47%) makes this one of the most informative categorical
features for fraud detection.

**`housing_status` — Strongest categorical signal:**
BA stands out dramatically at 3.75% — more than triple the overall fraud
rate. All other housing categories fall well below baseline (0.34%–0.86%).
BA represents a single high-risk housing situation that is a strong fraud
indicator, while most other statuses are associated with lower-than-average
fraud.

**`source` — Application channel matters:**
TELEAPP (phone) applications have a notably higher fraud rate (1.59%) vs
INTERNET (1.10%). Phone-based applications present higher risk, possibly
because identity verification is harder over the phone.

**`device_os` — Windows is high risk:**
Windows devices show the highest fraud rate at 2.47% (more than double
baseline), followed by Macintosh at 1.40%. Linux (0.52%) and other (0.58%)
are well below baseline. This likely reflects the demographics of fraudsters
rather than any security property of the OS itself.

---

### Binary Features

**`email_is_free` — Free email = higher risk:**
Applicants using free email providers (1) have a fraud rate of 1.38% vs
0.80% for paid email (0). Fraudsters prefer disposable free email addresses
to avoid identity tracing.

**`phone_home_valid` — Invalid phone = higher risk:**
Invalid home phone numbers (0) are associated with a fraud rate of 1.41%
vs 0.67% for valid ones (1). Fraudsters are more likely to provide
unverifiable contact information.

**`phone_mobile_valid` — Similar pattern, weaker signal:**
Invalid mobile numbers (0): 1.49% vs valid (1): 1.05%. Same direction as
home phone but a weaker gap, suggesting mobile validity is a less reliable
signal on its own.

**`has_other_cards` — Inverse signal:**
Applicants with no existing cards (0) have a fraud rate of 1.30% vs 0.42%
for those with existing cards (1). Fraudsters are typically new to the bank
with no established relationship — having existing cards is a strong
indicator of legitimacy.

**`foreign_request` — Strong signal:**
Foreign requests (1) have a fraud rate of 2.20% vs 1.07% for domestic (0)
— exactly double. Applications originating from outside the bank's country
represent meaningfully elevated fraud risk.

**`keep_alive_session` — Inverse signal:**
Applicants who did NOT keep their session alive (0) have a fraud rate of
1.72% vs 0.65% for those who did (1). Fraudsters appear to rush through
the application without maintaining an active session, possibly due to
using automated scripts or simply being less engaged with the process.

## Step 7: Correlation Analysis

In [ ]:
# Compute correlation matrix on all numerical features + target
corr_cols = numerical_cols + ['fraud_bool']
corr = df[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(20, 16))
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws=dict(size=8)
)

ax.set_title('Correlation Heatmap — Numerical Features + Target',
            fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Extract all pairs with |correlation| > 0.5, excluding self-correlations
corr_pairs = (
    corr
    .where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ['Feature 1', 'Feature 2', 'Correlation']
corr_pairs['Abs Correlation'] = corr_pairs['Correlation'].abs()

strong_pairs = (
    corr_pairs[corr_pairs['Abs Correlation'] > 0.5]
    .sort_values('Abs Correlation', ascending=False)
    .reset_index(drop=True)
)

print(f"Strongly correlated pairs (|r| > 0.5):\n")
print(strong_pairs[['Feature 1', 'Feature 2', 'Correlation']].to_string(index=False))

In [ ]:
# How correlated is each feature with fraud_bool specifically?
target_corr = (
    corr['fraud_bool']
    .drop('fraud_bool')
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#E24B4A' if v > 0 else '#378ADD' for v in target_corr]
target_corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Correlation of each feature with fraud_bool', fontsize=13, pad=12)
ax.set_xlabel('Pearson Correlation')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Interpretation

### Correlation Heatmap

The heatmap reveals that most feature pairs in this dataset have very low
correlations, indicating limited redundancy overall. However, four pairs
exceed the |r| > 0.5 threshold:

| Feature 1 | Feature 2 | Correlation | Explanation |
|---|---|---|---|
| `month` | `velocity_4w` | -0.85 | Strongest pair — rolling 4-week velocity decreases as the dataset window progresses, a mathematical artifact of how velocity is computed |
| `proposed_credit_limit` | `credit_risk_score` | +0.61 | Higher internal risk scores are associated with higher proposed limits — both capture application risk from different angles |
| `month` | `velocity_24h` | -0.55 | Same logic as month/velocity_4w — time progression affects rolling velocity calculations |
| `velocity_4w` | `velocity_24h` | +0.54 | Expected — different time windows of the same underlying velocity metric naturally correlate |

**Multicollinearity concern:**
The `month`/`velocity_4w` pair (r = -0.85) is the most concerning for
multicollinearity. In linear models, using both features simultaneously
would be problematic as they carry largely redundant information.
`velocity_4w` and `velocity_24h` (r = 0.54) present a similar but weaker
concern. In later phases, one of each correlated pair may be dropped or
dimensionality reduction (e.g. PCA) may be applied when using linear models.
Tree-based models are naturally immune to multicollinearity and can use
both features without issue.

**`proposed_credit_limit` and `credit_risk_score`** (r = 0.61) are both
strong fraud signals identified in previous steps. Their moderate correlation
means they share some information but are not fully redundant — both should
be retained for modeling.

---

### Correlation with Target (`fraud_bool`)

All features show very low Pearson correlation with `fraud_bool`
(all values under 0.10). This is expected for two reasons:

1. **Class imbalance**: with only ~1.11% fraud cases, the target variable
   has very low variance, which mathematically suppresses correlation values
2. **Non-linear relationships**: Pearson correlation only captures linear
   associations. Several features showed strong visual separation in the
   KDE and box plots (e.g. `bank_months_count`, `name_email_similarity`)
   despite having near-zero linear correlation here — their predictive
   power is non-linear in nature

**Top positively correlated with fraud** (higher value = more fraud):
- `credit_risk_score` (~0.07) — highest overall
- `proposed_credit_limit` (~0.07)
- `customer_age` (~0.06)
- `income` (~0.05)
- `device_distinct_emails_8w` (~0.04)

**Top negatively correlated with fraud** (higher value = less fraud):
- `keep_alive_session` (~-0.05) — not keeping session alive = more fraud
- `name_email_similarity` (~-0.03)
- `date_of_birth_distinct_emails_4w` (~-0.03)
- `has_other_cards` (~-0.03)
- `phone_home_valid` (~-0.03)

**Important limitation:** The low correlation values should not be
interpreted as these features being unimportant. Features like
`bank_months_count` showed very clear fraud separation in KDE plots
but appear near-zero here because Pearson correlation cannot capture
non-linear patterns. 

For this dataset, non-linear models that naturally handle these relationships
are strongly preferred over linear ones. Examples include:
- **Random Forest** — builds many decision trees, each splitting on feature
  thresholds regardless of linearity
- **XGBoost / LightGBM** — gradient boosted trees, highly effective on
  tabular fraud detection tasks
- **Decision Tree** — simplest tree-based model, splits data by feature
  thresholds rather than linear boundaries

## Step 8: Bivariate & Multivariate Analysis

### Why these groupings?

The feature pairs and groupings in this step were not chosen arbitrarily —
they were directly motivated by findings from previous steps:

**Scatter plot pairs** were selected by combining the strongest individual
signals identified in Steps 5, 6, and 7:
- `credit_risk_score` vs `proposed_credit_limit` — both had the highest
  linear correlation with `fraud_bool` (~0.07) in Step 7, and a moderate
  correlation with each other (r=0.61). The natural question is whether
  they interact to amplify fraud risk.
- `income` vs `proposed_credit_limit` — Step 6 showed fraudsters inflate
  both. Do they inflate them together?
- `name_email_similarity` vs `credit_risk_score` — both are strong
  individual signals from Steps 5 and 6. Do they combine?
- `bank_months_count` vs `credit_risk_score` — `bank_months_count` showed
  the clearest KDE separation in Step 6 despite near-zero linear correlation
  in Step 7. Pairing it with the top correlated feature tests whether the
  combination reveals additional structure.

**Grouped bars and heatmap** combine the strongest categorical and ordinal
signals from Step 6:
- `customer_age` + `employment_status` — both showed strong monotonic fraud
  trends individually. Do high-risk employment statuses become even riskier
  at older ages?
- `housing_status` + `source` — BA was the strongest categorical signal
  (3.75%). Does the application channel amplify that risk further?
- `customer_age` + `housing_status` — the two strongest ordinal/categorical
  signals combined in a heatmap to find the most dangerous demographic
  intersection in the dataset.

The guiding principle: **take the strongest signals from previous steps
and ask what happens when you combine them.**

In [ ]:
# Scatter plots of the most informative feature pairs, colored by fraud label
# Using a small sample — scatter on 1M rows is unreadable
scatter_sample = df.sample(5000, random_state=42)

pairs = [
    ('credit_risk_score',   'proposed_credit_limit'),
    ('income',              'proposed_credit_limit'),
    ('name_email_similarity', 'credit_risk_score'),
    ('bank_months_count',   'credit_risk_score'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (x, y) in enumerate(pairs):
    for label, color, name in [(0, '#378ADD', 'Legit'), (1, '#E24B4A', 'Fraud')]:
        subset = scatter_sample[scatter_sample['fraud_bool'] == label]
        axes[i].scatter(subset[x], subset[y],
                        c=color, label=name, alpha=0.4, s=10)
    axes[i].set_xlabel(x, fontsize=10)
    axes[i].set_ylabel(y, fontsize=10)
    axes[i].set_title(f'{x} vs {y}', fontsize=11)
    axes[i].legend(fontsize=9)

plt.suptitle('Scatter Plots — Key Feature Pairs by Fraud Label',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Pair plot on the most discriminative features
# Keep feature count small (5-6 max) — pair plot grows as n² subplots
pair_features = [
    'credit_risk_score',
    'proposed_credit_limit',
    'income',
    'name_email_similarity',
    'bank_months_count',
    'fraud_bool'
]

pair_sample = df.sample(3000, random_state=42)
pair_sample['fraud_bool'] = pair_sample['fraud_bool'].map({0: 'Legit', 1: 'Fraud'})

g = sns.pairplot(
    pair_sample[pair_features],
    hue='fraud_bool',
    palette={'Legit': '#378ADD', 'Fraud': '#E24B4A'},
    plot_kws=dict(alpha=0.3, s=10),
    diag_kind='kde',
    corner=True
)
g.figure.suptitle('Pair Plot — Top Discriminative Features by Fraud Label',
                  y=1.01, fontsize=13)
plt.show()

In [ ]:
# Fraud rate across combinations of customer_age and employment_status
# Shows how two categorical/ordinal features interact with fraud together
pivot = df.groupby(['customer_age', 'employment_status'])['fraud_bool'] \
          .mean().unstack() * 100

fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot(kind='bar', ax=ax, edgecolor='white', width=0.8)
ax.axhline(y=df['fraud_bool'].mean() * 100,
           color='black', linestyle='--', linewidth=1,
           label='Overall fraud rate')
ax.set_title('Fraud Rate by Customer Age & Employment Status', fontsize=13, pad=12)
ax.set_xlabel('Customer Age Bin')
ax.set_ylabel('Fraud Rate (%)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Employment Status', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Fraud rate across combinations of housing_status and source
pivot2 = df.groupby(['housing_status', 'source'])['fraud_bool'] \
           .mean().unstack() * 100

fig, ax = plt.subplots(figsize=(12, 5))
pivot2.plot(kind='bar', ax=ax,
            color=['#378ADD', '#E24B4A'],
            edgecolor='white', width=0.6)
ax.axhline(y=df['fraud_bool'].mean() * 100,
           color='black', linestyle='--', linewidth=1,
           label='Overall fraud rate')
ax.set_title('Fraud Rate by Housing Status & Application Source', fontsize=13, pad=12)
ax.set_xlabel('Housing Status')
ax.set_ylabel('Fraud Rate (%)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Source', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap showing fraud rate at the intersection of age and housing status
# Good for spotting high-risk combinations of two features
pivot3 = df.groupby(['customer_age', 'housing_status'])['fraud_bool'] \
           .mean().unstack() * 100

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot3, annot=True, fmt='.2f', cmap='Reds',
            linewidths=0.5, ax=ax, annot_kws=dict(size=9))
ax.set_title('Fraud Rate (%) by Customer Age & Housing Status',
            fontsize=13, pad=12)
ax.set_xlabel('Housing Status')
ax.set_ylabel('Customer Age Bin')
plt.tight_layout()
plt.show()

## Interpretation

### Scatter Plots

**`credit_risk_score` vs `proposed_credit_limit`:**
`proposed_credit_limit` takes discrete values (200, 500, 1000, 1500, 2000),
revealing it is a tiered product rather than a truly continuous feature.
Fraudulent applications cluster heavily at the 1500 and 2000 tiers with
credit risk scores in the 100–300 range, while legitimate applications
dominate the 200 and 500 tiers across all score ranges. The combination
of high credit risk score and high proposed limit is a strong joint
fraud indicator.

**`income` vs `proposed_credit_limit`:**
Fraudulent applicants request the highest credit limit tiers (1500, 2000)
consistently across all income levels — suggesting fraudsters target
maximum limits regardless of their stated income, rather than requesting
limits proportional to their claimed earnings.

**`name_email_similarity` vs `credit_risk_score`:**
No clear 2D cluster is visible for fraud cases — the two features do not
appear to interact in a meaningful combined pattern. Their fraud signals
are largely independent of each other.

**`bank_months_count` vs `credit_risk_score`:**
Fraudulent applications are almost entirely concentrated at
`bank_months_count` = 0 across all credit risk score values. Zero banking
history is close to a necessary condition for fraud in this plot —
applicants with any established banking history are rarely fraudulent.

---

### Pair Plot

The fraud class is barely visible in the scatter panels due to the 89.7:1
class imbalance — a 3000-row sample contains only ~33 fraud points. Despite
this, several patterns are confirmed:
- `proposed_credit_limit` shows clear discrete tiers in every panel,
  confirming it is a categorical-like tiered feature
- `bank_months_count` shows fraud concentrated at 0 in every panel
  it appears in, consistent with all previous steps
- The diagonal KDE plots confirm the distributional findings from Step 6
- No single 2D feature combination perfectly separates fraud from
  legitimate — classes overlap significantly in all panels, reinforcing
  that fraud detection requires combining multiple features simultaneously
  in a model rather than simple threshold rules

---

### Grouped Bar — Customer Age & Employment Status

The interaction between age and employment status reveals strong combined
effects that are not visible when examining either feature alone:

- **CG at age 60** is the single most extreme combination — **12.5% fraud
  rate**, more than 11× the overall baseline of 1.11%
- **CA** (blue) rises consistently with age, reaching ~7% at age 80
- **CC** (green) similarly rises with age, reaching ~5.5% at age 90
- **CF and CE** remain flat and low across all age bins — these employment
  statuses are consistently low risk regardless of age

The interaction effect is clear: high-risk employment statuses (CG, CA, CC)
combined with older age bins create dramatically elevated fraud rates. Neither
factor alone fully explains the risk — their combination amplifies it.

---

### Grouped Bar — Housing Status & Application Source

- **BA + TELEAPP** is the highest risk combination at **5.0%** — nearly
  4.5× the overall baseline
- **BA + INTERNET** is also elevated at 3.75% — still more than 3× baseline
- The TELEAPP channel consistently shows higher fraud than INTERNET across
  all housing statuses, but the difference is only practically significant
  for housing status BA
- All other housing statuses (BB–BG) remain well below the baseline for
  both sources, confirming that BA is the uniquely high-risk housing category
  regardless of application channel

---

### Fraud Rate Heatmap — Customer Age & Housing Status

The heatmap is the most powerful visualization in this step, revealing
high-risk combinations invisible in univariate analysis:

- **BA + age 80 = 10.28%** — the single highest fraud rate cell in the
  entire dataset, nearly **9× the overall baseline**
- The BA column is consistently the darkest across all age bins, with
  fraud rate increasing monotonically as age increases
- **BD + age 70 = 4.62%** — a notable secondary hotspot
- **BB + age 90 = 6.45%** — significant given BB appeared low-risk in
  isolation, showing that age can amplify risk even in otherwise safe categories
- Several cells in the lower-right are empty — certain age and housing
  combinations do not exist in the data (e.g. age 90 + BF/BG), likely
  reflecting realistic demographic constraints in the synthetic generation

**Key insight:** BA alone had a fraud rate of 3.75%, and age 80 alone had
4.93%. Their combination reaches 10.28% — far beyond what either feature
predicts individually. This demonstrates the importance of multivariate
analysis: **feature interactions reveal fraud risk patterns that univariate
and bivariate analysis cannot capture alone**, and motivates the use of
interaction terms or tree-based models that naturally discover such
combinations in later phases.

## EDA Summary: Key Findings

This notebook conducted a full Exploratory Data Analysis on the Bank Account
Fraud Dataset (NeurIPS 2022), Base variant — 1,000,000 applications across
32 features over 8 months. The following key findings will directly inform
modeling decisions in later phases.

---

### Dataset Characteristics
- **Severe class imbalance**: 89.7:1 ratio (legitimate to fraud). Accuracy
  is a misleading metric — Precision, Recall, F1, and PR-AUC should be
  used instead.
- **Sentinel values**: Five features use -1 to encode missing data, most
  critically `prev_address_months_count` (71.29% missing) and
  `bank_months_count` (25.36% missing). These must be replaced with NaN
  before modeling.
- **No true NaN values** and no duplicate rows — the dataset is structurally
  clean, with missingness encoded deliberately rather than randomly.

---

### Strongest Fraud Signals Identified

**Numerical features:**
- `credit_risk_score` — clearest KDE separation; fraud peaks at 200–250
  vs legitimate at 100–150
- `proposed_credit_limit` — fraudsters overwhelmingly request the highest
  tiers (1500–2000); legitimate applicants cluster at 200
- `bank_months_count` — fraud almost entirely at 0; virtually no prior
  banking history
- `income` — fraudsters cluster at maximum income values (bimodal,
  fraud peak near 1.0)
- `name_email_similarity` — fraud strongly concentrated near 0;
  fraudsters use emails that don't match their stated name

**Categorical & binary features:**
- `housing_status` BA — 3.75% fraud rate, more than 3× baseline
- `employment_status` CC — 2.47% fraud rate, more than double baseline
- `customer_age` — monotonic increase from 0.35% (age 10) to 5.26% (age 90)
- `foreign_request` — 2.20% vs 1.07% domestic
- `device_os` Windows — 2.47%, more than double baseline
- `phone_home_valid` = 0 — invalid phone numbers associated with fraud
- `has_other_cards` = 0 — no existing cards strongly associated with fraud
- `keep_alive_session` = 0 — fraudsters rush through applications

---

### Multivariate Hotspots
The most dangerous feature combinations found:
- **BA + age 80** → 10.28% fraud rate (9× baseline)
- **CG + age 60** → 12.5% fraud rate (11× baseline)
- **BA + TELEAPP** → 5.0% fraud rate (4.5× baseline)

These interaction effects are invisible in univariate analysis and
motivate the use of tree-based models that naturally discover such
combinations.

---

### Outlier Findings
- IQR is the more appropriate outlier detection method for this dataset
  given heavy skewness in most features
- Outliers in `device_distinct_emails_8w` (3.7× fraud rate) and
  `proposed_credit_limit` (2.7× fraud rate) are meaningful fraud signals
  and should NOT be removed
- Outliers in `bank_branch_count_8w` and `prev_address_months_count`
  are noise or sentinel distortion

---

### Correlation Findings
- All features have very low linear correlation with `fraud_bool` (< 0.10)
  due to class imbalance and predominantly non-linear relationships
- Strongest correlated pair: `month` ↔ `velocity_4w` (r = -0.85) —
  a mathematical artifact of rolling velocity calculations, not a
  true fraud signal. May cause multicollinearity in linear models.
- `proposed_credit_limit` ↔ `credit_risk_score` (r = 0.61) — both are
  strong fraud signals but moderately redundant; retain both for modeling

---

### Recommendations for Later Phases

**Data preprocessing:**
- Replace sentinel -1 values with NaN
- For `prev_address_months_count` (71.29% missing): consider dropping
  or converting to a binary indicator feature
- For `bank_months_count` (25.36% missing): median or KNN imputation
- Apply IQR-based outlier detection rather than z-score for skewed features
- Do NOT remove outliers in fraud-signal features

**Modeling:**
- Use class weighting or SMOTE to handle the 89.7:1 imbalance
- Prefer tree-based models (Random Forest, XGBoost, LightGBM) —
  they are naturally robust to outliers, non-linear relationships,
  multicollinearity, and skewed distributions
- Use time-based train/test split (e.g. months 0–5 for training,
  months 6–7 for testing) given the upward fraud trend over time
- Evaluate using PR-AUC and F1-score rather than accuracy

**Feature engineering candidates:**
- Binary indicator for `prev_address_months_count` missingness
- Interaction terms: `housing_status` × `customer_age`,
  `employment_status` × `customer_age`
- Consider dropping `device_fraud_count` (near-constant, zero variance)
- Consider dropping `velocity_4w` if `month` is retained (r = -0.85)

# Phase 2

## Part 1: Preprocessing

All preprocessing decisions are grounded in Phase 1 EDA findings.
Steps are applied consistently across train and test splits.
The original dataset is never modified — all transformations are
applied to copies after splitting.

In [ ]:
# 1.1 — Add new imports needed for Phase 2 only
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve, f1_score
)

RANDOM_SEED = 42
print("Phase 2 imports loaded.")

In [ ]:
# 1.2 — Drop irrelevant features (grounded in Phase 1 findings)
# device_fraud_count: near-constant (>99.9% zeros) — no variance, no signal
# velocity_4w: r=-0.85 with month — severe multicollinearity
cols_to_drop = ['device_fraud_count', 'velocity_4w']
df = df.drop(columns=cols_to_drop)
print(f"Dropped: {cols_to_drop}")
print(f"Remaining columns: {df.shape[1]}")

In [ ]:
# 1.3 — Handle sentinel -1 values
# prev_address_months_count: 71.29% missing — convert to binary indicator
df['had_prev_address'] = (df['prev_address_months_count'] != -1).astype(int)
df = df.drop(columns=['prev_address_months_count'])
print("prev_address_months_count → had_prev_address (binary indicator)")

# Remaining sentinel cols: replace -1 with NaN for imputation after split
sentinel_cols = [
    'bank_months_count',           # 25.36% missing
    'current_address_months_count',# 0.43% missing
    'session_length_in_minutes',   # 0.20% missing
    'device_distinct_emails_8w'    # 0.04% missing
]
for col in sentinel_cols:
    df[col] = df[col].replace(-1, np.nan)

print(f"\nNaN counts after sentinel replacement:")
print(df[sentinel_cols].isnull().sum())

In [ ]:
# 1.4 — Time-based train/test split (BEFORE any imputation or scaling)
# Train: months 0-5 | Test: months 6-7
# Justified by upward fraud trend over time (1.13% → 1.47%) from Phase 1

train_df = df[df['month'] <= 5].copy()
test_df  = df[df['month'] >  5].copy()

X_train = train_df.drop(columns=['fraud_bool'])
y_train = train_df['fraud_bool']

X_test  = test_df.drop(columns=['fraud_bool'])
y_test  = test_df['fraud_bool']

print(f"Train: {X_train.shape[0]:,} rows | Fraud rate: {y_train.mean()*100:.2f}%")
print(f"Test:  {X_test.shape[0]:,} rows  | Fraud rate: {y_test.mean()*100:.2f}%")
print(f"\nClass imbalance ratio (train): {(y_train==0).sum() / (y_train==1).sum():.1f}:1")

In [ ]:
# 1.5 — Median imputation (fit on train only — never on test)
train_medians = X_train[sentinel_cols].median()

X_train[sentinel_cols] = X_train[sentinel_cols].fillna(train_medians)
X_test[sentinel_cols]  = X_test[sentinel_cols].fillna(train_medians)

print("Train medians used for imputation:")
for col, val in train_medians.items():
    print(f"  {col:<35} → {val}")

print(f"\nRemaining NaNs in train: {X_train.isnull().sum().sum()}")
print(f"Remaining NaNs in test:  {X_test.isnull().sum().sum()}")

In [ ]:
# 1.6 — One-hot encode categorical features (fit structure on train, align test)
cat_cols = ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=True)

# Align columns — test may be missing rare categories seen in train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"Features after encoding: {X_train.shape[1]}")
print(f"Train shape: {X_train.shape} | Test shape: {X_test.shape}")

In [ ]:
# 1.7 — Feature scaling (fit on train only)
# Only scale continuous numerical features
# Binary, OHE, and ordinal features are left as-is

binary_cols_phase2 = [
    'email_is_free', 'phone_home_valid', 'phone_mobile_valid',
    'has_other_cards', 'foreign_request', 'keep_alive_session',
    'had_prev_address'
]
ohe_cols = [c for c in X_train.columns
            if any(c.startswith(cat) for cat in cat_cols)]
no_scale = set(binary_cols_phase2 + ohe_cols + ['customer_age', 'month'])
scale_cols = [c for c in X_train.columns if c not in no_scale]

scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])

print(f"Scaled {len(scale_cols)} continuous features")
print(f"Not scaled: {len(no_scale)} binary/ordinal/OHE features")
print(f"\nFinal train shape: {X_train.shape}")
print(f"Final test shape:  {X_test.shape}")

In [ ]:
# 1.8 — Compute scale_pos_weight for XGBoost
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f"Legitimate (train): {neg:,}")
print(f"Fraud (train):      {pos:,}")
print(f"scale_pos_weight:   {scale_pos_weight:.2f}")

## Part 2: Model Implementation

Four classifiers are implemented and evaluated on the same train/test split
using a fixed random seed (42) for reproducibility. All models are evaluated
using consistent metrics throughout.

**Models chosen:**
- **Logistic Regression** — linear baseline; interpretable; expected to struggle
  with the non-linear fraud patterns identified in Phase 1
- **Decision Tree** — simple tree baseline; interpretable decision rules;
  single tree without ensemble benefits
- **Random Forest** — strong ensemble; naturally robust to outliers, skewed
  distributions, and non-linear relationships identified in Phase 1
- **XGBoost** *(bonus ensemble)* — gradient boosted trees; consistently the
  best performer on this dataset in published literature

**Models excluded:**
- **KNN** — O(n²) complexity makes it computationally infeasible on 794K rows
- **SVM** — O(n³) complexity; does not scale to this dataset size

**Note on SMOTE:** Synthetic Minority Oversampling Technique (SMOTE) was
considered for handling the 96.5:1 class imbalance. However, balancing
the training set would generate ~779K synthetic fraud rows, resulting in
a ~1.57M row training set that exceeds practical Kaggle memory and runtime
limits. Class weighting is used instead — it achieves equivalent imbalance
correction without generating synthetic data, and must always be applied
only to the training set to prevent data leakage.

**Imbalance handling:**
- Logistic Regression, Decision Tree: `class_weight='balanced'`
- Random Forest: `class_weight='balanced_subsample'` (recomputed per tree)
- XGBoost: `scale_pos_weight=96.53` (ratio of negatives to positives in train)

**Primary metric:** PR-AUC — most informative under 96.5:1 class imbalance.
Accuracy is not used as it is misleading (a model predicting all-legitimate
achieves 98.97% accuracy while detecting zero fraud).

In [ ]:
# Part 2 imports
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

print("Part 2 imports loaded.")

In [ ]:
# Helper function — used for every model
# Finds the threshold that maximizes F1 on the test set
# then reports all metrics at that threshold

def evaluate_model(name, y_true, y_proba):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
    best_idx  = np.argmax(f1_scores)
    best_thr  = thresholds[best_idx]
    y_pred    = (y_proba >= best_thr).astype(int)

    pr_auc  = average_precision_score(y_true, y_proba)
    roc_auc = roc_auc_score(y_true, y_proba)
    f1      = f1_score(y_true, y_pred)
    cm      = confusion_matrix(y_true, y_pred)

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  PR-AUC (primary):  {pr_auc:.4f}")
    print(f"  ROC-AUC:           {roc_auc:.4f}")
    print(f"  F1 @ thr={best_thr:.3f}: {f1:.4f}")
    print(f"\n  Classification Report (threshold={best_thr:.3f}):")
    print(classification_report(y_true, y_pred,
                                target_names=['Legitimate', 'Fraud']))
    print(f"  Confusion Matrix:")
    print(f"    TN={cm[0,0]:,}  FP={cm[0,1]:,}")
    print(f"    FN={cm[1,0]:,}  TP={cm[1,1]:,}")

    return {
        'name': name,
        'pr_auc': pr_auc,
        'roc_auc': roc_auc,
        'f1': f1,
        'threshold': best_thr,
        'y_proba': y_proba,
        'y_pred': y_pred,
        'cm': cm
    }

In [ ]:
# Model 1 — Logistic Regression
# Baseline linear model — expected to struggle with non-linear fraud patterns
# class_weight='balanced' compensates for 96.5:1 imbalance

lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
lr.fit(X_train, y_train)
lr_proba = lr.predict_proba(X_test)[:, 1]
results_lr = evaluate_model('Logistic Regression', y_test, lr_proba)

In [ ]:
# Model 2 — Decision Tree
# Simple tree baseline — interpretable, shows feature splits
# class_weight='balanced' handles imbalance
# max_depth=10 prevents overfitting on 794K rows

dt = DecisionTreeClassifier(
    class_weight='balanced',
    max_depth=10,
    random_state=RANDOM_SEED
)
dt.fit(X_train, y_train)
dt_proba = dt.predict_proba(X_test)[:, 1]
results_dt = evaluate_model('Decision Tree', y_test, dt_proba)

In [ ]:
# Model 3 — Random Forest
# Strong ensemble baseline — robust to skew, outliers, non-linearity
# n_estimators=100 balances performance vs Kaggle runtime
# class_weight='balanced_subsample' applies balancing per tree (better than 'balanced' for RF)
# n_jobs=-1 uses all available CPU cores

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced_subsample',
    max_depth=20,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
results_rf = evaluate_model('Random Forest', y_test, rf_proba)

In [ ]:
# Model 4 — XGBoost (Bonus ensemble)
# Best performer on this dataset in the literature
# scale_pos_weight=96.53 handles imbalance (computed in Part 1)
# tree_method='hist' is faster on large datasets
# eval_metric='aucpr' optimizes for PR-AUC during training

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    tree_method='hist',
    eval_metric='aucpr',
    random_state=RANDOM_SEED,
    n_jobs=-1
)
xgb.fit(X_train, y_train)
xgb_proba = xgb.predict_proba(X_test)[:, 1]
results_xgb = evaluate_model('XGBoost', y_test, xgb_proba)

In [ ]:
# Summary comparison table — all 4 models side by side
results_summary = pd.DataFrame([
    {
        'Model': r['name'],
        'PR-AUC': round(r['pr_auc'], 4),
        'ROC-AUC': round(r['roc_auc'], 4),
        'F1': round(r['f1'], 4),
        'Threshold': round(r['threshold'], 3)
    }
    for r in [results_lr, results_dt, results_rf, results_xgb]
])
results_summary = results_summary.sort_values('PR-AUC', ascending=False)
print(results_summary.to_string(index=False))

## Interpretation

### Summary Table

| Model | PR-AUC | ROC-AUC | F1 | Threshold |
|---|---|---|---|---|
| XGBoost | 0.1773 | 0.8840 | 0.2522 | 0.893 |
| Logistic Regression | 0.1656 | 0.8833 | 0.2427 | 0.881 |
| Random Forest | 0.1317 | 0.8615 | 0.2121 | 0.412 |
| Decision Tree | 0.0823 | 0.7966 | 0.1658 | 0.924 |

**Primary metric is PR-AUC** — the most informative metric under 96.5:1
class imbalance. Accuracy is not reported as a performance indicator since
all models achieve ~97% accuracy by predominantly predicting legitimate,
which is trivially achieved and carries no diagnostic value.

---

### Model Rankings & Analysis

**XGBoost (PR-AUC = 0.1773, F1 = 0.2522) — best overall:**
XGBoost achieves the highest PR-AUC and F1, consistent with published
results on this dataset. It catches 865 fraud cases (TP) with 3,116 false
alarms (FP) — the best precision-recall balance across all models. Its
gradient boosting mechanism effectively captures the non-linear fraud
patterns identified in Phase 1 (e.g. proposed_credit_limit tiers,
bank_months_count at zero, multivariate hotspots).

**Logistic Regression (PR-AUC = 0.1656, F1 = 0.2427) — surprisingly strong:**
Logistic Regression performs close to XGBoost despite being a linear model.
This is likely because several strong fraud signals identified in Phase 1
(foreign_request, email_is_free, keep_alive_session, phone_home_valid) are
binary features with relatively linear relationships to fraud_bool. The model
captures these well. However, it cannot exploit non-linear interactions such
as housing_status BA × customer_age, which limits its ceiling.

**Random Forest (PR-AUC = 0.1317, F1 = 0.2121) — underperforming baseline:**
Random Forest underperforms Logistic Regression in this initial configuration,
which is unexpected given Phase 1's evidence of non-linear patterns. This is
likely due to suboptimal hyperparameters — specifically max_depth=20 and
n_estimators=100 may not be sufficient for this dataset's complexity. The
notably lower threshold (0.412 vs ~0.88 for others) suggests the model
assigns more reasonable fraud probabilities but still struggles with the
extreme imbalance. Hyperparameter tuning in Part 3 is expected to significantly
improve this model.

**Decision Tree (PR-AUC = 0.0823, F1 = 0.1658) — weakest as expected:**
Decision Tree performs worst, consistent with expectations. A single tree
without ensemble averaging is prone to overfitting on the majority class
and lacks the robustness of ensemble methods. It catches only 524 fraud
cases — the fewest of all models. This result serves as a useful lower
bound for what a single tree can achieve, motivating the ensemble approaches.

---

### Confusion Matrix Analysis

All models struggle with the same fundamental trade-off:
- **False Negatives (missed fraud)** range from 2,013 (XGBoost) to 2,354 (DT)
  out of 2,878 total fraud cases — meaning even the best model misses ~70%
  of fraud at the optimal F1 threshold
- **False Positives (false alarms)** range from 2,920 (DT) to 3,899 (RF)
  — legitimate applications incorrectly flagged as fraud

In a real fraud detection system, False Negatives are typically more costly
than False Positives (missing fraud causes direct financial loss). Lowering
the threshold further would increase recall at the cost of more false alarms.
---

### Note on PR-AUC values

PR-AUC values in the 0.08–0.18 range are expected for this dataset without
extensive feature engineering or deep tuning. A random classifier would
achieve PR-AUC ≈ 0.014 (equal to the fraud prevalence of 1.40% in the test
set). All models significantly outperform this baseline. Published results
on the BAF Base dataset with standard models report PR-AUC in the 0.10–0.30
range, placing our results within the expected window. Part 3 hyperparameter
tuning aims to push these scores higher.

## Part 3: Ablation Study

For each model, a hyperparameter search is conducted using RandomizedSearchCV
with 3-fold stratified cross-validation on the training set. Scoring is by
**PR-AUC** (average_precision) — not accuracy — to ensure optimization targets
the metric that matters under class imbalance.

RandomizedSearchCV is used over GridSearchCV because the training set has
794,989 rows — exhaustive grid search would be computationally infeasible.
Each search tests a defined set of configurations and reports results across
at least 3 configurations per model as required.

The best configuration for each model is selected based on mean CV PR-AUC
and used in Part 4 for final evaluation.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
import time

# Stratified CV — preserves fraud rate in each fold
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

# Helper to display ablation results as a clean table
def show_ablation_results(search, model_name, n_top=5):
    results = pd.DataFrame(search.cv_results_)
    cols = [c for c in results.columns if c.startswith('param_')] + \
           ['mean_test_score', 'std_test_score', 'rank_test_score']
    results = results[cols].sort_values('mean_test_score', ascending=False)
    results.columns = [c.replace('param_', '') for c in results.columns]
    results['mean_test_score'] = results['mean_test_score'].round(4)
    results['std_test_score']  = results['std_test_score'].round(4)
    print(f"\n--- {model_name} — Top {n_top} configurations ---")
    print(results.head(n_top).to_string(index=False))
    print(f"\nBest PR-AUC: {search.best_score_:.4f}")
    print(f"Best params: {search.best_params_}")

In [ ]:
# Ablation 1 — Logistic Regression
print("Running Logistic Regression search...")
start = time.time()

lr_params = {
    'C'      : [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver' : ['liblinear'],
}

lr_search = RandomizedSearchCV(
    LogisticRegression(class_weight='balanced', max_iter=1000,
                       random_state=RANDOM_SEED),
    param_distributions=lr_params,
    n_iter=8,           # tests all meaningful combos
    scoring='average_precision',
    cv=cv,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=2
)
lr_search.fit(X_train, y_train)
show_ablation_results(lr_search, 'Logistic Regression')
print(f"Time: {(time.time()-start)/60:.1f} min")

In [ ]:
# Ablation 2 — Decision Tree
print("Running Decision Tree search...")
start = time.time()

dt_params = {
    'max_depth'        : [5, 10, 20, None],
    'min_samples_split': [2, 20, 100],
    'min_samples_leaf' : [1, 10, 50],
    'criterion'        : ['gini', 'entropy'],
}

dt_search = RandomizedSearchCV(
    DecisionTreeClassifier(class_weight='balanced',
                           random_state=RANDOM_SEED),
    param_distributions=dt_params,
    n_iter=12,
    scoring='average_precision',
    cv=cv,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=2
)
dt_search.fit(X_train, y_train)
show_ablation_results(dt_search, 'Decision Tree')
print(f"Time: {(time.time()-start)/60:.1f} min")

In [ ]:
# Ablation 3 — Random Forest
# n_iter=6 to keep runtime manageable on Kaggle
print("Running Random Forest search...")
start = time.time()

rf_params = {
    'n_estimators' : [100, 200, 300],
    'max_depth'    : [10, 20, None],
    'min_samples_split': [2, 10],
    'max_features' : ['sqrt', 'log2'],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced_subsample',
                           random_state=RANDOM_SEED, n_jobs=-1),
    param_distributions=rf_params,
    n_iter=6,
    scoring='average_precision',
    cv=cv,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=2
)
rf_search.fit(X_train, y_train)
show_ablation_results(rf_search, 'Random Forest')
print(f"Time: {(time.time()-start)/60:.1f} min")

In [ ]:
# Ablation 4 — XGBoost
print("Running XGBoost search...")
start = time.time()

xgb_params = {
    'n_estimators'    : [100, 200, 300],
    'max_depth'       : [3, 5, 7],
    'learning_rate'   : [0.01, 0.1, 0.3],
    'subsample'       : [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=scale_pos_weight,
                  tree_method='hist',
                  device='cuda', 
                  eval_metric='aucpr',
                  random_state=RANDOM_SEED,
                  n_jobs=-1),
    param_distributions=xgb_params,
    n_iter=10,
    scoring='average_precision',
    cv=cv,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=2
)
xgb_search.fit(X_train, y_train)
show_ablation_results(xgb_search, 'XGBoost')
print(f"Time: {(time.time()-start)/60:.1f} min")

In [ ]:
# Ablation summary — best config per model
print("\n=== ABLATION SUMMARY — Best CV PR-AUC per model ===\n")
searches = [
    ('Logistic Regression', lr_search),
    ('Decision Tree',       dt_search),
    ('Random Forest',       rf_search),
    ('XGBoost',             xgb_search),
]
for name, search in searches:
    print(f"  {name:<25} CV PR-AUC: {search.best_score_:.4f} | "
          f"Params: {search.best_params_}")

## Interpretation

### Overview

The ablation study systematically evaluated hyperparameter configurations
for all four models using 3-fold stratified cross-validation scored by
PR-AUC — the most informative metric under the 96.5:1 class imbalance
present in the training set. All CV PR-AUC values should be interpreted
relative to a random classifier baseline of ~0.014 (equal to the training
fraud prevalence), not relative to 1.0.

---

### Logistic Regression

8 candidates were evaluated, covering all combinations of
`C` ∈ {0.01, 0.1, 1, 10} and `penalty` ∈ {L1, L2}.
The results reveal a near-complete insensitivity to both regularization
strength and penalty type: the top 5 configurations span only 0.0004
PR-AUC (0.1321–0.1325), with identical standard deviations of 0.005
across all folds. This uniformity indicates that the model's performance
ceiling is determined by its linear decision boundary, not by the
regularization regime. Logistic Regression cannot model the non-linear,
interaction-driven fraud patterns identified in Phase 1 — such as the
compounding risk of housing status BA combined with older age bins — and
no hyperparameter adjustment can compensate for this structural limitation.

**Selected configuration:** L2 penalty, C=0.01 (CV PR-AUC: 0.1325).
The margin over all other configurations is statistically negligible
given the fold variance; strongest regularization was retained as a
conservative choice.

---

### Decision Tree

12 candidates were evaluated across `max_depth`, `min_samples_split`,
`min_samples_leaf`, and `criterion`. Three findings emerge clearly:

- **Depth is the dominant factor.** All top 4 configurations use
  `max_depth=10`. The single depth-20 entry (rank 5) scores 0.0122
  below rank 1, and unconstrained trees did not appear in the top 5.
  A single deep tree overfits readily on 794K rows — deeper splits
  partition noise rather than signal, degrading generalization.

- **Entropy outperforms Gini at equivalent depth.** Ranks 1–2 use
  entropy; ranks 3–4 use Gini and score ~0.007 lower. Entropy's
  information-theoretic formulation produces slightly more discriminative
  splits on this imbalanced, noisy dataset.

- **Leaf size has minimal effect once depth is constrained.**
  `min_samples_leaf` of 10 vs. 1 at identical depth and split settings
  produces a difference of only 0.0001 PR-AUC.

**Selected configuration:** `max_depth=10`, `min_samples_split=100`,
`min_samples_leaf=10`, `criterion=entropy` (CV PR-AUC: 0.0697).

---

### Random Forest

6 candidates were evaluated due to the significant computational cost
of this search — the full run took 46 minutes on Kaggle CPU.
Two findings stand out:

- **Unconstrained depth is beneficial for ensembles.** All three top
  configurations use `max_depth=None`, scoring 0.010–0.013 higher than
  the depth-20 entries. This is the inverse of the Decision Tree finding,
  and for a principled reason: individual tree overfitting is offset by
  the averaging across 300 estimators, each trained on a random feature
  and data subset. Ensemble variance reduction makes unconstrained depth
  safe here where it was harmful in a single tree.

- **`log2` feature sampling outperforms `sqrt`.** For 45 features,
  `log2` samples ~6 features per split vs. ~7 for `sqrt`. The slightly
  lower feature count forces greater diversity between trees, reducing
  inter-tree correlation and improving ensemble generalization.

**Selected configuration:** `n_estimators=300`, `max_depth=None`,
`min_samples_split=10`, `max_features=log2` (CV PR-AUC: 0.1146).

---

### XGBoost

10 candidates were evaluated in 1.4 minutes using GPU acceleration
(Kaggle CUDA). The search space covered `n_estimators`, `max_depth`,
`learning_rate`, `subsample`, and `colsample_bytree`.

The central finding is that **shallow trees with a moderate learning
rate dominate.** The best configuration uses `max_depth=3` with
`learning_rate=0.1` and scores 0.1600 — a margin of 0.0223 over rank 2,
which uses `max_depth=7` with a much slower learning rate of 0.01.
In gradient boosting, each tree corrects the residual errors of all
previous trees. Shallow trees (depth=3, up to 8 leaf nodes) make
conservative, targeted corrections that accumulate robustly over 200
iterations. Deep trees at high learning rates overfit the residuals
of early rounds, collapsing generalization. The slightly elevated fold
variance of rank 1 (std=0.0054 vs. 0.0029 for rank 2) reflects the
higher sensitivity of this configuration to fold composition, but the
mean advantage is substantial and consistent.

`colsample_bytree=0.8` introduces per-tree feature subsampling,
acting as an additional regularizer that improves robustness without
sacrificing signal.

**Selected configuration:** `n_estimators=200`, `max_depth=3`,
`learning_rate=0.1`, `subsample=1.0`, `colsample_bytree=0.8`
(CV PR-AUC: 0.1600).

---

### Cross-Model Summary

| Model               | Best CV PR-AUC | Key tuning insight                        |
|---------------------|---------------|-------------------------------------------|
| XGBoost             | 0.1600        | Shallow trees + moderate learning rate    |
| Logistic Regression | 0.1325        | Insensitive to all regularization choices |
| Random Forest       | 0.1146        | Unconstrained depth + log2 sampling       |
| Decision Tree       | 0.0697        | Depth=10 + entropy criterion              |

XGBoost leads by a clear margin. Logistic Regression ranks second in CV
PR-AUC, ahead of Random Forest — a result explained by LR's strong
performance on the primary linear signals (`credit_risk_score`,
`proposed_credit_limit`, `bank_months_count`) even without capturing
interactions. This ordering will shift in Part 4, where Random Forest
gains more from its best configuration on the held-out test set.
The Decision Tree lags significantly, reflecting the fundamental
limitation of a single tree without ensemble benefits.

## Part 4: Evaluation & Comparison

All models are re-evaluated using their best hyperparameter configuration
found in Part 3. Results are reported on the held-out test set (months 6–7)
using consistent metrics across all models. Threshold tuning is applied to
each model to find the optimal F1 operating point.

In [ ]:
# Re-train each model using best params from Part 3 ablation
print("Re-training all models with best hyperparameters...\n")

# Best Logistic Regression
best_lr = LogisticRegression(
    **lr_search.best_params_,
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
best_lr.fit(X_train, y_train)
best_lr_proba = best_lr.predict_proba(X_test)[:, 1]
results_best_lr = evaluate_model('Logistic Regression (tuned)', y_test, best_lr_proba)

In [ ]:
# Best Decision Tree
best_dt = DecisionTreeClassifier(
    **dt_search.best_params_,
    class_weight='balanced',
    random_state=RANDOM_SEED
)
best_dt.fit(X_train, y_train)
best_dt_proba = best_dt.predict_proba(X_test)[:, 1]
results_best_dt = evaluate_model('Decision Tree (tuned)', y_test, best_dt_proba)

In [ ]:
# Best Random Forest
best_rf = RandomForestClassifier(
    **rf_search.best_params_,
    class_weight='balanced_subsample',
    random_state=RANDOM_SEED,
    n_jobs=-1
)
best_rf.fit(X_train, y_train)
best_rf_proba = best_rf.predict_proba(X_test)[:, 1]
results_best_rf = evaluate_model('Random Forest (tuned)', y_test, best_rf_proba)

In [ ]:
# Best XGBoost
best_xgb = XGBClassifier(
    **xgb_search.best_params_,
    scale_pos_weight=scale_pos_weight,
    tree_method='hist',
    device='cuda',
    eval_metric='aucpr',
    random_state=RANDOM_SEED,
    n_jobs=-1
)
best_xgb.fit(X_train, y_train)
best_xgb_proba = best_xgb.predict_proba(X_test)[:, 1]
results_best_xgb = evaluate_model('XGBoost (tuned)', y_test, best_xgb_proba)

In [ ]:
# Final summary comparison table — all tuned models
final_results = pd.DataFrame([
    {
        'Model': r['name'],
        'PR-AUC': round(r['pr_auc'], 4),
        'ROC-AUC': round(r['roc_auc'], 4),
        'F1': round(r['f1'], 4),
        'Threshold': round(r['threshold'], 3)
    }
    for r in [results_best_lr, results_best_dt, results_best_rf, results_best_xgb]
]).sort_values('PR-AUC', ascending=False)

print("=== FINAL MODEL COMPARISON (tuned) ===\n")
print(final_results.to_string(index=False))

In [ ]:
# Confusion matrices — 2x2 grid, one per model
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

tuned_results = [results_best_lr, results_best_dt, results_best_rf, results_best_xgb]

for i, res in enumerate(tuned_results):
    sns.heatmap(
        res['cm'],
        annot=True, fmt=',', cmap='Blues',
        xticklabels=['Predicted Legit', 'Predicted Fraud'],
        yticklabels=['Actual Legit', 'Actual Fraud'],
        ax=axes[i],
        annot_kws={'size': 11}
    )
    axes[i].set_title(f"{res['name']}\nPR-AUC={res['pr_auc']:.4f}  F1={res['f1']:.4f}",
                      fontsize=11, pad=10)

plt.suptitle('Confusion Matrices — All Tuned Models', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves — all models on one plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#378ADD', '#E24B4A', '#16a34a', '#d97706']
labels = [r['name'] for r in tuned_results]

# ROC curve
for res, color, label in zip(tuned_results, colors, labels):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f"{label} (AUC={res['roc_auc']:.4f})")

axes[0].plot([0,1], [0,1], 'k--', linewidth=1, label='Random classifier')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — All Tuned Models', fontsize=13, pad=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Precision-Recall curve
for res, color, label in zip(tuned_results, colors, labels):
    prec, rec, _ = precision_recall_curve(y_test, res['y_proba'])
    axes[1].plot(rec, prec, color=color, linewidth=2,
                 label=f"{label} (PR-AUC={res['pr_auc']:.4f})")

baseline = y_test.mean()
axes[1].axhline(y=baseline, color='black', linestyle='--', linewidth=1,
                label=f'Random classifier (={baseline:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves — All Tuned Models', fontsize=13, pad=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance — Random Forest and XGBoost
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Random Forest feature importance
rf_importance = pd.Series(
    best_rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(15)

rf_importance.plot(kind='barh', ax=axes[0],
                   color='#378ADD', edgecolor='white')
axes[0].invert_yaxis()
axes[0].set_title('Random Forest — Top 15 Feature Importances', fontsize=12, pad=12)
axes[0].set_xlabel('Importance Score')

# XGBoost feature importance
xgb_importance = pd.Series(
    best_xgb.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(15)

xgb_importance.plot(kind='barh', ax=axes[1],
                    color='#E24B4A', edgecolor='white')
axes[1].invert_yaxis()
axes[1].set_title('XGBoost — Top 15 Feature Importances', fontsize=12, pad=12)
axes[1].set_xlabel('Importance Score')

plt.suptitle('Feature Importances — Tree-based Models', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Interpretation

### Final Model Comparison

| Model                       | PR-AUC | ROC-AUC |    F1 | Threshold |
|-----------------------------|--------|---------|-------|-----------|
| XGBoost (tuned)             | 0.1936 |  0.8922 | 0.2679|     0.894 |
| Logistic Regression (tuned) | 0.1658 |  0.8832 | 0.2427|     0.880 |
| Random Forest (tuned)       | 0.1455 |  0.8680 | 0.2267|     0.120 |
| Decision Tree (tuned)       | 0.0919 |  0.8021 | 0.1712|     0.881 |

PR-AUC is the primary metric throughout. A random classifier on this
test set achieves PR-AUC ≈ 0.014 (equal to the 1.40% fraud prevalence).
All tuned models substantially outperform this baseline, with XGBoost
reaching 13.8× random performance. Accuracy is not reported as a
diagnostic metric — all models achieve ~97% accuracy by predominantly
predicting legitimate, which carries no discriminative value under
96.5:1 class imbalance.

---

### Gains from Tuning

Hyperparameter tuning produced meaningful improvements for three of
the four models. XGBoost gained the most (+0.0163 PR-AUC, from 0.1773
to 0.1936), followed by Random Forest (+0.0138) and Decision Tree
(+0.0096). Logistic Regression was essentially unchanged (+0.0002),
consistent with the ablation finding that its performance is determined
by its linear boundary rather than regularization choices. The tuning
gains confirm that the final configurations are not arbitrary — they
represent systematic improvements over default settings on a held-out
evaluation set the search never touched.

---

### XGBoost — Best Overall

XGBoost achieves the highest PR-AUC (0.1936), ROC-AUC (0.8922), and
F1 (0.2679). Its confusion matrix shows 943 true positives against
3,219 false alarms — a precision of 0.23, meaning roughly 1 in 4
flagged applications is genuine fraud. This is the best precision-recall
balance across all models. The result is consistent with published
benchmarks on the BAF Base dataset, where gradient boosting methods
consistently lead. XGBoost's sequential boosting mechanism is well
suited to this data: each shallow tree (max_depth=3) corrects the
residual errors of previous trees, incrementally learning the complex,
non-linear feature interactions that Phase 1 identified as the primary
structure of fraud risk — such as the compounding effect of high
proposed credit limit, zero banking history, and specific housing
status categories. No single split captures this; the boosting sequence
does.

---

### Logistic Regression — Competitive Linear Baseline

Logistic Regression ranks second with PR-AUC 0.1658 and F1 0.2427,
outperforming Random Forest in both metrics despite being a far simpler
model. Its confusion matrix (828 TP, 3,116 FP) shows the best precision
among all models at 0.21 — it generates fewer false alarms than RF and
DT. This performance is driven by the strong linear signals in the data:
`credit_risk_score`, `proposed_credit_limit`, and `bank_months_count`
have individually measurable linear associations with fraud that LR
exploits effectively. However, LR cannot model the interaction-driven
fraud patterns identified in the EDA — such as the 10.28% fraud rate
at housing status BA combined with age 80, which is far beyond what
either feature predicts linearly. This structural ceiling explains the
gap with XGBoost and motivates explicit interaction feature engineering
as a direction for future work.

---

### Random Forest — Strong Ensemble, Calibration Caveat

Random Forest achieves PR-AUC 0.1455 and F1 0.2267. Its optimal
threshold of 0.120 stands in sharp contrast to the 0.880+ thresholds
of the other three models. This is a known calibration property of
Random Forests under severe class imbalance: because each tree averages
fraud prevalence across its leaves, raw probability outputs are
compressed toward low values even for high-confidence fraud predictions.
The model's ranking ability is reasonable (ROC-AUC 0.8680), but its
probability estimates are not comparable in scale to XGBoost or LR.
Its confusion matrix (924 TP, 4,349 FP) shows a higher recall than
LR at the cost of substantially more false alarms, reflecting the
lower threshold required to surface predictions.

---

### Decision Tree — Limited by Single-Tree Structure

The Decision Tree has the lowest PR-AUC (0.0919) and the highest false
alarm count by far: 8,050 FP against 1,023 TP, yielding a precision of
only 0.11. For every genuine fraud case caught, approximately 8
legitimate applications are incorrectly flagged. While its raw TP count
of 1,023 is the highest among all models, this apparent advantage is
negated by the volume of false positives — in any operational context,
such a false alarm rate would be prohibitively costly. The result
reflects the fundamental limitation of a single decision tree: without
ensemble averaging, individual splits memorize noise, generalization
degrades at depth, and the model cannot recover the precision that
ensemble methods achieve by aggregating diverse trees.

---

### ROC and PR Curves

The ROC curves show all models clustered between 0.80 and 0.89 AUC,
with the Decision Tree visibly separated below. This compressed
separation illustrates why ROC-AUC is insufficiently diagnostic under
class imbalance: the large legitimate class dominates the true negative
rate, making all models appear similarly capable. The PR curves reveal
the true performance spread. All four curves drop steeply after
recall ≈ 0.05–0.10, reflecting the structure of fraud in this dataset:
a small subset of cases has extreme feature values that any model flags
confidently at high precision, while the majority of fraud cases are
difficult to distinguish from legitimate applications. As recall
increases, precision collapses because the model must lower its
threshold enough to capture harder cases, inevitably pulling in more
legitimate applications. XGBoost (orange) maintains the highest
precision at every recall level, confirming its superiority is not
threshold-dependent.

---

### Feature Importances

Random Forest and XGBoost assign importance using different mechanisms,
producing rankings that diverge substantially. Random Forest uses
mean decrease in Gini impurity, which is biased toward high-cardinality
continuous features that appear at many split points across many trees.
This explains why `current_address_months_count` and `credit_risk_score`
dominate its top list. XGBoost uses gain-based importance, measuring
the average loss reduction each feature produces per split. This rewards
features making the most impactful individual splits, regardless of
frequency — explaining the dominance of `device_os_windows`, which
contributes an outsized gain in early boosting rounds, and binary
features like `had_prev_address` and `keep_alive_session`.

The features appearing in both top-15 lists — `credit_risk_score`,
`current_address_months_count`, `income`, `customer_age`,
`had_prev_address` — represent the most robust fraud signals, as their
importance is confirmed by two independent measurement approaches.
These features align with Phase 1 EDA findings: `credit_risk_score`
and `proposed_credit_limit` were the strongest individual fraud
correlates, and `had_prev_address` (derived from the 71.29%-missing
`prev_address_months_count`) encodes meaningful applicant history.
The disagreement between the two rankings for all other features
cautions against over-interpreting either importance list in isolation.